In [1]:
from scipy.io import loadmat

data = loadmat("DREAMER.mat")
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])


In [24]:
import pandas as pd
import numpy as np

print(data.keys())

dreamer = data["DREAMER"]

print(type(dreamer))
print(dreamer.shape)

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])
<class 'numpy.ndarray'>
(1, 1)


In [42]:
subjects = dreamer[0,0]["Data"][0]

print("Number of subjects:", len(subjects))


Number of subjects: 23


In [63]:
subject0 = subjects[0]

print("Valence shape:",
      subject0["ScoreValence"][0,0].shape)

print("Arousal shape:",
      subject0["ScoreArousal"][0,0].shape)

print("Dominance shape:",
      subject0["ScoreDominance"][0,0].shape)

print("\nValence scores:")
print(subject0["ScoreValence"][0,0].flatten())

Valence shape: (18, 1)
Arousal shape: (18, 1)
Dominance shape: (18, 1)

Valence scores:
[4 3 5 4 4 1 5 1 1 5 4 4 4 3 2 3 1 3]


In [64]:
eeg = subject0["EEG"][0,0]

print(eeg.dtype)

print("\nBaseline shape:")
print(eeg["baseline"][0,0].shape)

print("\nStimuli shape:")
print(eeg["stimuli"][0,0].shape)

[('baseline', 'O'), ('stimuli', 'O')]

Baseline shape:
(18, 1)

Stimuli shape:
(18, 1)


In [65]:
stimuli = eeg["stimuli"][0,0]

print(type(stimuli))

print("Number of videos:", stimuli.shape[1])

video0 = stimuli[0,0]

print("Video 1 EEG shape:")
print(video0.shape)

<class 'numpy.ndarray'>
Number of videos: 1
Video 1 EEG shape:
(25472, 14)


In [66]:
subject0 = subjects[0]

eeg = subject0["EEG"][0,0]

stimuli = eeg["stimuli"][0,0]

print("stimuli shape:", stimuli.shape)
print("stimuli dtype:", stimuli.dtype)

stimuli shape: (18, 1)
stimuli dtype: object


In [67]:
print(type(stimuli[0,0]))
print(stimuli[0,0].shape)

<class 'numpy.ndarray'>
(25472, 14)


In [69]:
for i in range(5):
    print(f"\nVideo {i+1}")
    print(stimuli[i,0].shape)


Video 1
(25472, 14)

Video 2
(16768, 14)

Video 3
(44544, 14)

Video 4
(21248, 14)

Video 5
(17408, 14)


In [70]:
all_eeg = []
all_labels = []
all_subject_ids = []

for subject_idx in range(len(subjects)):

    subject = subjects[subject_idx]

    eeg = subject["EEG"][0,0]
    stimuli = eeg["stimuli"][0,0]

    arousal_scores = subject["ScoreArousal"][0,0].flatten()

    for video_idx in range(18):

        eeg_signal = stimuli[video_idx,0]

        arousal = arousal_scores[video_idx]

        all_eeg.append(eeg_signal)
        all_labels.append(arousal)
        all_subject_ids.append(subject_idx)

print("Total recordings:", len(all_eeg))
print("Total labels:", len(all_labels))
print("Total subject IDs:", len(all_subject_ids))

Total recordings: 414
Total labels: 414
Total subject IDs: 414


In [71]:
print(np.unique(all_labels))
print("Min Label:", np.min(all_labels))
print("Max Label:", np.max(all_labels))

[1 2 3 4 5]
Min Label: 1
Max Label: 5


In [72]:
from collections import Counter
import numpy as np

binary_labels = np.where(np.array(all_labels) > 3, 1, 0)

print(Counter(binary_labels))

Counter({np.int64(0): 233, np.int64(1): 181})


In [73]:
import numpy as np

WINDOW_SIZE = 128
STEP_SIZE = 64

X_windows = []
y_windows = []
groups = []

for eeg_signal, label, subject_id in zip(
        all_eeg,
        binary_labels,
        all_subject_ids):

    n_samples = eeg_signal.shape[0]

    for start in range(
            0,
            n_samples - WINDOW_SIZE + 1,
            STEP_SIZE):

        end = start + WINDOW_SIZE

        window = eeg_signal[start:end]

        X_windows.append(window)
        y_windows.append(label)
        groups.append(subject_id)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows)
groups = np.array(groups)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("groups:", groups.shape)

X_windows: (171074, 128, 14)
y_windows: (171074,)
groups: (171074,)


In [74]:
print(np.unique(groups))
print("Number of Subjects:", len(np.unique(groups)))

print(np.bincount(y_windows))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
Number of Subjects: 23
[89077 81997]


In [75]:
import os
import numpy as np

os.makedirs("Dataset/Processed", exist_ok=True)

np.save("Dataset/Processed/X_windows.npy", X_windows)
np.save("Dataset/Processed/y_windows.npy", y_windows)
np.save("Dataset/Processed/groups.npy", groups)

In [76]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X_windows,
        y_windows,
        groups=groups
    )
)

X_train = X_windows[train_idx]
X_test = X_windows[test_idx]

y_train = y_windows[train_idx]
y_test = y_windows[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train subjects:",
      len(np.unique(groups_train)))

print("Test subjects:",
      len(np.unique(groups_test)))

print("Common subjects:",
      np.intersect1d(
          np.unique(groups_train),
          np.unique(groups_test)
      ))

Train shape: (133884, 128, 14)
Test shape: (37190, 128, 14)
Train subjects: 18
Test subjects: 5
Common subjects: []


In [77]:
import numpy as np

def extract_statistical_features(window):

    features = []

    for ch in range(window.shape[1]):

        signal = window[:, ch]

        features.extend([
            np.mean(signal),
            np.std(signal),
            np.min(signal),
            np.max(signal),
            np.var(signal),
            np.sqrt(np.mean(signal**2))
        ])

    return features

In [78]:
X_train_features = np.array([ extract_statistical_features(window) for window in X_train ])
print(X_train_features.shape)

(133884, 84)


In [79]:
X_test_features = np.array([ extract_statistical_features(window)for window in X_test ])
print(X_test_features.shape)

(37190, 84)


In [80]:
np.save("Dataset/Processed/X_train_features.npy",
        X_train_features)

np.save("Dataset/Processed/X_test_features.npy",
        X_test_features)

In [81]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import time

start = time.time()

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_features, y_train)

rf_time = time.time() - start

y_pred_rf = rf.predict(X_test_features)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

print("\nTraining Time:", rf_time)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Accuracy : 0.5451734337187416
Precision: 0.5549786019971469
Recall   : 0.5161872412695043
F1 Score : 0.5348805235515715

Training Time: 133.4995334148407

Confusion Matrix:
[[10549  7799]
 [ 9116  9726]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.57      0.56     18348
           1       0.55      0.52      0.53     18842

    accuracy                           0.55     37190
   macro avg       0.55      0.55      0.54     37190
weighted avg       0.55      0.55      0.54     37190



In [82]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": range(X_train_features.shape[1]),
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(20))

    Feature  Importance
34       34    0.020106
19       19    0.019819
22       22    0.019777
35       35    0.019679
30       30    0.019613
31       31    0.019546
33       33    0.018659
7         7    0.017864
10       10    0.017659
0         0    0.017608
67       67    0.017208
5         5    0.017071
68       68    0.017004
70       70    0.016954
6         6    0.016736
13       13    0.016636
11       11    0.016228
9         9    0.016218
1         1    0.015614
23       23    0.015435


In [83]:
import numpy as np

X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

for ch in range(X_train.shape[2]):

    mean = X_train[:,:,ch].mean()
    std = X_train[:,:,ch].std()

    X_train_norm[:,:,ch] = (
        X_train[:,:,ch] - mean
    ) / (std + 1e-8)

    X_test_norm[:,:,ch] = (
        X_test[:,:,ch] - mean
    ) / (std + 1e-8)

print(X_train_norm.shape)
print(X_test_norm.shape)

(133884, 128, 14)
(37190, 128, 14)


In [86]:
from scipy.signal import welch
import numpy as np

def bandpower(signal, fs, low, high):

    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=min(len(signal),128)
    )

    idx = np.logical_and(
        freqs >= low,
        freqs <= high
    )

    return np.trapezoid(psd[idx], freqs[idx])

In [87]:
def extract_features(window, fs=128):

    features = []

    for ch in range(window.shape[1]):

        sig = window[:,ch]

        # Statistical
        features.extend([
            np.mean(sig),
            np.std(sig),
            np.min(sig),
            np.max(sig),
            np.var(sig),
            np.sqrt(np.mean(sig**2))
        ])

        # Frequency bands

        delta = bandpower(sig, fs, 0.5, 4)
        theta = bandpower(sig, fs, 4, 8)
        alpha = bandpower(sig, fs, 8, 13)
        beta  = bandpower(sig, fs, 13, 30)
        gamma = bandpower(sig, fs, 30, 45)

        features.extend([
            delta,
            theta,
            alpha,
            beta,
            gamma
        ])

    return features

In [ ]:
X_train_psd = np.array([
    extract_features(window)
    for window in X_train_norm
])

print(X_train_psd.shape)

(20000, 128, 14)


In [ ]:
from scipy.io import loadmat

data = loadmat("DREAMER.mat")
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])


In [ ]:
import pandas as pd
import numpy as np

print(data.keys())

dreamer = data["DREAMER"]

print(type(dreamer))
print(dreamer.shape)

dict_keys(['__header__', '__version__', '__globals__', 'DREAMER'])
<class 'numpy.ndarray'>
(1, 1)


In [ ]:
subjects = dreamer[0,0]["Data"][0]

print("Number of subjects:", len(subjects))


Number of subjects: 23


In [ ]:
subject0 = subjects[0]

print("Valence shape:",
      subject0["ScoreValence"][0,0].shape)

print("Arousal shape:",
      subject0["ScoreArousal"][0,0].shape)

print("Dominance shape:",
      subject0["ScoreDominance"][0,0].shape)

print("\nValence scores:")
print(subject0["ScoreValence"][0,0].flatten())

Valence shape: (18, 1)
Arousal shape: (18, 1)
Dominance shape: (18, 1)

Valence scores:
[4 3 5 4 4 1 5 1 1 5 4 4 4 3 2 3 1 3]


In [ ]:
eeg = subject0["EEG"][0,0]

print(eeg.dtype)

print("\nBaseline shape:")
print(eeg["baseline"][0,0].shape)

print("\nStimuli shape:")
print(eeg["stimuli"][0,0].shape)

[('baseline', 'O'), ('stimuli', 'O')]

Baseline shape:
(18, 1)

Stimuli shape:
(18, 1)


In [ ]:
stimuli = eeg["stimuli"][0,0]

print(type(stimuli))

print("Number of videos:", stimuli.shape[1])

video0 = stimuli[0,0]

print("Video 1 EEG shape:")
print(video0.shape)

<class 'numpy.ndarray'>
Number of videos: 1
Video 1 EEG shape:
(25472, 14)


In [ ]:
subject0 = subjects[0]

eeg = subject0["EEG"][0,0]

stimuli = eeg["stimuli"][0,0]

print("stimuli shape:", stimuli.shape)
print("stimuli dtype:", stimuli.dtype)

stimuli shape: (18, 1)
stimuli dtype: object


In [ ]:
print(type(stimuli[0,0]))
print(stimuli[0,0].shape)

<class 'numpy.ndarray'>
(25472, 14)


In [ ]:
for i in range(5):
    print(f"\nVideo {i+1}")
    print(stimuli[i,0].shape)


Video 1
(25472, 14)

Video 2
(16768, 14)

Video 3
(44544, 14)

Video 4
(21248, 14)

Video 5
(17408, 14)


In [ ]:
all_eeg = []
all_labels = []
all_subject_ids = []

for subject_idx in range(len(subjects)):

    subject = subjects[subject_idx]

    eeg = subject["EEG"][0,0]
    stimuli = eeg["stimuli"][0,0]

    arousal_scores = subject["ScoreArousal"][0,0].flatten()

    for video_idx in range(18):

        eeg_signal = stimuli[video_idx,0]

        arousal = arousal_scores[video_idx]

        all_eeg.append(eeg_signal)
        all_labels.append(arousal)
        all_subject_ids.append(subject_idx)

print("Total recordings:", len(all_eeg))
print("Total labels:", len(all_labels))
print("Total subject IDs:", len(all_subject_ids))

Total recordings: 414
Total labels: 414
Total subject IDs: 414


In [ ]:
print(np.unique(all_labels))
print("Min Label:", np.min(all_labels))
print("Max Label:", np.max(all_labels))

[1 2 3 4 5]
Min Label: 1
Max Label: 5


In [ ]:
from collections import Counter
import numpy as np

binary_labels = np.where(np.array(all_labels) > 3, 1, 0)

print(Counter(binary_labels))

Counter({np.int64(0): 233, np.int64(1): 181})


In [ ]:
import numpy as np

WINDOW_SIZE = 128
STEP_SIZE = 64

X_windows = []
y_windows = []
groups = []

for eeg_signal, label, subject_id in zip(
        all_eeg,
        binary_labels,
        all_subject_ids):

    n_samples = eeg_signal.shape[0]

    for start in range(
            0,
            n_samples - WINDOW_SIZE + 1,
            STEP_SIZE):

        end = start + WINDOW_SIZE

        window = eeg_signal[start:end]

        X_windows.append(window)
        y_windows.append(label)
        groups.append(subject_id)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows)
groups = np.array(groups)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("groups:", groups.shape)

X_windows: (171074, 128, 14)
y_windows: (171074,)
groups: (171074,)


In [ ]:
print(np.unique(groups))
print("Number of Subjects:", len(np.unique(groups)))

print(np.bincount(y_windows))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
Number of Subjects: 23
[89077 81997]


In [ ]:
import os
import numpy as np

os.makedirs("Dataset/Processed", exist_ok=True)

np.save("Dataset/Processed/X_windows.npy", X_windows)
np.save("Dataset/Processed/y_windows.npy", y_windows)
np.save("Dataset/Processed/groups.npy", groups)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X_windows,
        y_windows,
        groups=groups
    )
)

X_train = X_windows[train_idx]
X_test = X_windows[test_idx]

y_train = y_windows[train_idx]
y_test = y_windows[test_idx]

groups_train = groups[train_idx]
groups_test = groups[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Train subjects:",
      len(np.unique(groups_train)))

print("Test subjects:",
      len(np.unique(groups_test)))

print("Common subjects:",
      np.intersect1d(
          np.unique(groups_train),
          np.unique(groups_test)
      ))

Train shape: (133884, 128, 14)
Test shape: (37190, 128, 14)
Train subjects: 18
Test subjects: 5
Common subjects: []


In [ ]:
import numpy as np

def extract_statistical_features(window):

    features = []

    for ch in range(window.shape[1]):

        signal = window[:, ch]

        features.extend([
            np.mean(signal),
            np.std(signal),
            np.min(signal),
            np.max(signal),
            np.var(signal),
            np.sqrt(np.mean(signal**2))
        ])

    return features

In [ ]:
X_train_features = np.array([ extract_statistical_features(window) for window in X_train ])
print(X_train_features.shape)

(133884, 84)


In [ ]:
X_test_features = np.array([ extract_statistical_features(window)for window in X_test ])
print(X_test_features.shape)

(37190, 84)


In [ ]:
np.save("Dataset/Processed/X_train_features.npy",
        X_train_features)

np.save("Dataset/Processed/X_test_features.npy",
        X_test_features)

In [103]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import time

start = time.time()

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_features, y_train)

rf_time = time.time() - start

y_pred_rf = rf.predict(X_test_features)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("Model : Random Forest Classifier")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)

print("\nTraining Time:", rf_time)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Model : Random Forest Classifier
Accuracy : 0.5451734337187416
Precision: 0.5549786019971469
Recall   : 0.5161872412695043
F1 Score : 0.5348805235515715

Training Time: 63.68332600593567

Confusion Matrix:
[[10549  7799]
 [ 9116  9726]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.57      0.56     18348
           1       0.55      0.52      0.53     18842

    accuracy                           0.55     37190
   macro avg       0.55      0.55      0.54     37190
weighted avg       0.55      0.55      0.54     37190



In [ ]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": range(X_train_features.shape[1]),
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(20))

    Feature  Importance
34       34    0.020106
19       19    0.019819
22       22    0.019777
35       35    0.019679
30       30    0.019613
31       31    0.019546
33       33    0.018659
7         7    0.017864
10       10    0.017659
0         0    0.017608
67       67    0.017208
5         5    0.017071
68       68    0.017004
70       70    0.016954
6         6    0.016736
13       13    0.016636
11       11    0.016228
9         9    0.016218
1         1    0.015614
23       23    0.015435


In [ ]:
import numpy as np

X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

for ch in range(X_train.shape[2]):

    mean = X_train[:,:,ch].mean()
    std = X_train[:,:,ch].std()

    X_train_norm[:,:,ch] = (
        X_train[:,:,ch] - mean
    ) / (std + 1e-8)

    X_test_norm[:,:,ch] = (
        X_test[:,:,ch] - mean
    ) / (std + 1e-8)

print(X_train_norm.shape)
print(X_test_norm.shape)

(133884, 128, 14)
(37190, 128, 14)


In [ ]:
from scipy.signal import welch
import numpy as np

def bandpower(signal, fs, low, high):

    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=min(len(signal),128)
    )

    idx = np.logical_and(
        freqs >= low,
        freqs <= high
    )

    return np.trapezoid(psd[idx], freqs[idx])

In [ ]:
def extract_features(window, fs=128):

    features = []

    for ch in range(window.shape[1]):

        sig = window[:,ch]

        # Statistical
        features.extend([
            np.mean(sig),
            np.std(sig),
            np.min(sig),
            np.max(sig),
            np.var(sig),
            np.sqrt(np.mean(sig**2))
        ])

        # Frequency bands

        delta = bandpower(sig, fs, 0.5, 4)
        theta = bandpower(sig, fs, 4, 8)
        alpha = bandpower(sig, fs, 8, 13)
        beta  = bandpower(sig, fs, 13, 30)
        gamma = bandpower(sig, fs, 30, 45)

        features.extend([
            delta,
            theta,
            alpha,
            beta,
            gamma
        ])

    return features

In [90]:
import numpy as np

np.random.seed(42)

idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]

sample0 = np.random.choice(idx0, 10000, replace=False)
sample1 = np.random.choice(idx1, 10000, replace=False)

train_subset = np.concatenate([sample0, sample1])

X_train_small = X_train_norm[train_subset]
y_train_small = y_train[train_subset]

print(X_train_small.shape)

(20000, 128, 14)


In [91]:
idx0 = np.where(y_test == 0)[0]
idx1 = np.where(y_test == 1)[0]

sample0 = np.random.choice(idx0, 5000, replace=False)
sample1 = np.random.choice(idx1, 5000, replace=False)

test_subset = np.concatenate([sample0, sample1])

X_test_small = X_test_norm[test_subset]
y_test_small = y_test[test_subset]

print(X_test_small.shape)

(10000, 128, 14)


In [92]:
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)

print(y_train_cat.shape)
print(y_test_cat.shape)

(133884, 2)
(37190, 2)


In [93]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    Dropout,
    Flatten,
    Dense
)

cnn = Sequential([

    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        input_shape=(128,14)
    ),

    BatchNormalization(),

    MaxPooling1D(2),

    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    ),

    BatchNormalization(),

    MaxPooling1D(2),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.3),

    Dense(2, activation='softmax')
])

cnn.summary()

d:\VIT_Projects\dreamer\drenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 126, 64)        │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 61, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 520,130 (1.98 MB)

 Trainable params: 519,746 (1.98 MB)

 Non-trainable params: 384 (1.50 KB)

In [94]:
cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [95]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = cnn.fit(
    X_train_norm,
    y_train_cat,

    validation_split=0.2,

    epochs=15,
    batch_size=256,

    callbacks=[early_stop],

    verbose=1
)

Epoch 1/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 36s 84ms/step - accuracy: 0.6369 - loss: 0.6581 - val_accuracy: 0.5172 - val_loss: 0.8957
Epoch 2/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 55s 119ms/step - accuracy: 0.7031 - loss: 0.5463 - val_accuracy: 0.5548 - val_loss: 0.7963
Epoch 3/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 52s 125ms/step - accuracy: 0.7333 - loss: 0.5030 - val_accuracy: 0.5638 - val_loss: 0.7855
Epoch 4/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 43s 31ms/step - accuracy: 0.7518 - loss: 0.4708 - val_accuracy: 0.5516 - val_loss: 0.9149
Epoch 5/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.7694 - loss: 0.4457 - val_accuracy: 0.5651 - val_loss: 0.8482
Epoch 6/15
419/419 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.7812 - loss: 0.4279 - val_accuracy: 0.5702 - val_loss: 1.1309


In [96]:
loss, cnn_acc = cnn.evaluate(
    X_test_norm,
    y_test_cat,
    verbose=0
)

print("CNN Accuracy:", cnn_acc)

CNN Accuracy: 0.5096531510353088


In [99]:
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

y_pred = cnn.predict(X_test_norm)

y_pred = np.argmax(y_pred, axis=1)

print("CNN Model Performance Metric")
print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

print("Precision:",
      precision_score(y_test,y_pred))

print("Recall:",
      recall_score(y_test,y_pred))

print("F1:",
      f1_score(y_test,y_pred))

1163/1163 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
CNN Model Performance Metric
[[11846  6502]
 [11734  7108]]
              precision    recall  f1-score   support

           0       0.50      0.65      0.57     18348
           1       0.52      0.38      0.44     18842

    accuracy                           0.51     37190
   macro avg       0.51      0.51      0.50     37190
weighted avg       0.51      0.51      0.50     37190

Precision: 0.5222630418809698
Recall: 0.3772423309627428
F1: 0.43806236903734747


In [110]:
import numpy as np

np.random.seed(42)

idx0 = np.where(y_train == 0)[0]
idx1 = np.where(y_train == 1)[0]

sample0 = np.random.choice(idx0, 10000, replace=False)
sample1 = np.random.choice(idx1, 10000, replace=False)

subset = np.concatenate([sample0, sample1])

X_train_lstm = X_train_norm[subset]
y_train_lstm = y_train[subset]

print(X_train_lstm.shape)

(20000, 128, 14)


In [111]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

lstm_model = Sequential([
    LSTM(32, input_shape=(128,14)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(2, activation='softmax')
])

In [112]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [113]:
history = lstm_model.fit(
    X_train_lstm,
    to_categorical(y_train_lstm,2),
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    verbose=1
)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.5882 - loss: 0.6780 - val_accuracy: 0.0105 - val_loss: 0.9509
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.6255 - loss: 0.6567 - val_accuracy: 0.0132 - val_loss: 0.9493
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 174ms/step - accuracy: 0.6287 - loss: 0.6503 - val_accuracy: 0.0320 - val_loss: 0.9667
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.6304 - loss: 0.6447 - val_accuracy: 0.0593 - val_loss: 0.9444
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.6349 - loss: 0.6381 - val_accuracy: 0.1147 - val_loss: 0.9476
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.6386 - loss: 0.6312 - val_accuracy: 0.1780 - val_loss: 0.9635
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.6474 - loss: 0.6219 - val_accuracy: 0.3180 - val_loss: 0.8575
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.6544 - loss: 0.6179 - val_accuracy: 0.3540 - 

In [114]:
loss, lstm_acc = lstm_model.evaluate(
    X_test_norm,
    y_test_cat,
    verbose=0
)

print("LSTM Accuracy:", lstm_acc)

LSTM Accuracy: 0.49642378091812134


In [115]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

y_pred = lstm_model.predict(X_test_norm)
y_pred = np.argmax(y_pred, axis=1)


print("LSTM Model Performance")

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

print("Precision:",
      precision_score(y_test,y_pred))

print("Recall:",
      recall_score(y_test,y_pred))

print("F1:",
      f1_score(y_test,y_pred))

1163/1163 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step
LSTM Model Performance
[[16131  2217]
 [16511  2331]]
              precision    recall  f1-score   support

           0       0.49      0.88      0.63     18348
           1       0.51      0.12      0.20     18842

    accuracy                           0.50     37190
   macro avg       0.50      0.50      0.42     37190
weighted avg       0.50      0.50      0.41     37190

Precision: 0.512532981530343
Recall: 0.12371298163676892
F1: 0.19931594698589142


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

svm = LinearSVC(
    random_state=42,
    max_iter=5000
)

svm.fit(
    X_train_features,
    y_train
)

y_pred_svm = svm.predict(
    X_test_features
)

print("Accuracy:",
      accuracy_score(y_test,y_pred_svm))

print("Precision:",
      precision_score(y_test,y_pred_svm))

print("Recall:",
      recall_score(y_test,y_pred_svm))

print("F1:",
      f1_score(y_test,y_pred_svm))

print(confusion_matrix(
    y_test,
    y_pred_svm
))

print(classification_report(
    y_test,
    y_pred_svm
))

Accuracy: 0.47910728690508203
Precision: 0.4833479954756818
Recall: 0.4082369175246789
F1: 0.4426286108873288
[[10126  8222]
 [11150  7692]]
              precision    recall  f1-score   support

           0       0.48      0.55      0.51     18348
           1       0.48      0.41      0.44     18842

    accuracy                           0.48     37190
   macro avg       0.48      0.48      0.48     37190
weighted avg       0.48      0.48      0.48     37190



In [122]:
rf_results = {
    "Model": "Random Forest",
    "Accuracy": rf_accuracy * 100,
    "Precision": rf_precision * 100,
    "Recall": rf_recall * 100,
    "F1 Score": rf_f1 * 100
}

svm_results = {
    "Model": "SVM",
    "Accuracy": accuracy_score(y_test, y_pred_svm) * 100,
    "Precision": precision_score(y_test, y_pred_svm) * 100,
    "Recall": recall_score(y_test, y_pred_svm) * 100,
    "F1 Score": f1_score(y_test, y_pred_svm) * 100
}

cnn_results = {
    "Model": "CNN",
    "Accuracy": cnn_acc * 100,
    "Precision": precision_score(y_test, y_pred) * 100,
    "Recall": recall_score(y_test, y_pred) * 100,
    "F1 Score": f1_score(y_test, y_pred) * 100
}

lstm_results = {
    "Model": "LSTM",
    "Accuracy": lstm_acc * 100,
    "Precision": precision_score(y_test, y_pred) * 100,
    "Recall": recall_score(y_test, y_pred) * 100,
    "F1 Score": f1_score(y_test, y_pred) * 100
}

In [123]:
final_comparison = pd.DataFrame([
    rf_results,
    svm_results,
    cnn_results,
    lstm_results
])

final_comparison = final_comparison.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

final_comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,54.517343,55.497860,51.618724,53.488052
1,CNN,50.712556,51.253298,12.371298,19.931595
2,LSTM,49.642378,51.253298,12.371298,19.931595
3,SVM,47.910729,48.334800,40.823692,44.262861
